<a href="https://colab.research.google.com/github/vatsal-py-lab/f1-performance-analytics/blob/data-cleaning/f1_performance_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install fastf1 library

!pip install fastf1

In [2]:
# import libraries

import pkgutil
import fastf1
import pandas as pd
import os

In [3]:
# list modules programmatically

modules = [m.name for m in pkgutil.iter_modules(fastf1.__path__)]
print(modules)

['_api', '_version', 'api', 'core', 'ergast', 'events', 'internals', 'legacy', 'livetiming', 'logger', 'mvapi', 'plotting', 'req', 'signalr_aio', 'utils']


In [4]:
# sets up a cache directory
cache_dir = 'cache'
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# enables the fastf1 cache
fastf1.Cache.enable_cache(cache_dir)

In [5]:
# print columns using 'get_event_schedule' function for year 2025
season = 2025
schedule = fastf1.get_event_schedule(season)
print(schedule.columns)

Index(['RoundNumber', 'Country', 'Location', 'OfficialEventName', 'EventDate',
       'EventName', 'EventFormat', 'Session1', 'Session1Date',
       'Session1DateUtc', 'Session2', 'Session2Date', 'Session2DateUtc',
       'Session3', 'Session3Date', 'Session3DateUtc', 'Session4',
       'Session4Date', 'Session4DateUtc', 'Session5', 'Session5Date',
       'Session5DateUtc', 'F1ApiSupport'],
      dtype='object')


In [6]:
# creating data tables - events

today = pd.Timestamp.today().normalize()
schedule['Completed'] = schedule['EventDate'] < today
schedule['EventID'] = schedule['RoundNumber'] + 1
schedule['Year'] = "2025"
events = schedule[['EventID', 'Year', 'EventName', 'Country', 'Location', 'EventDate', 'EventFormat', 'Completed']]
events.to_csv('events.csv', index=False)

In [7]:
# print columns using 'get_session' function for Melbourne GP for year 2025

location = 'Melbourne'
session_type = 'R'
session = fastf1.get_session(season, location, session_type)
session.load()
print(session.laps.columns)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.0]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core   

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')


In [8]:
# creating data tables - drivers

driver = session.results[['Abbreviation', 'FullName', 'TeamName', 'DriverNumber']].copy()
driver.rename(columns={'Abbreviation': 'DriverCode',
                       'FullName': 'DriverName',
                       'TeamName': 'Team',
                       'DriverNumber': 'CarNumber'}, inplace=True)
driver['DriverId'] = range(1, len(driver) + 1)
driver = driver[['DriverId', 'DriverCode', 'DriverName', 'Team', 'CarNumber']]
driver.to_csv('drivers.csv', index=False)

In [9]:
# creating data tables - laps

laps = session.laps[['Driver', 'DriverNumber', 'LapNumber', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
                     'Position', 'Stint', 'Compound', 'TyreLife', 'PitInTime', 'PitOutTime']]
laps.insert(0, 'LapId', range(1, len(laps) + 1))
laps = pd.merge(laps, driver[['DriverId', 'DriverCode']], left_on='Driver', right_on='DriverCode', how='left')
laps = laps.drop(columns=['DriverCode'])
laps = laps[['DriverId', 'Driver', 'LapId', 'LapNumber', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
                     'Position', 'Stint', 'Compound', 'TyreLife', 'PitInTime', 'PitOutTime']]
laps.to_csv('laps.csv', index=False)

In [10]:
# creating data tables - weather

weather = session.weather_data
weather.insert(0, 'WeatherId', range(1, len(weather) + 1))
weather["Session"] = session_type
weather = weather[['WeatherId', 'Session', 'Time', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']]
weather.to_csv('weather.csv', index=False)

In [11]:
# creating data table - telemetry

tel = session.car_data
tel_list = list(tel.keys())
telemetry = pd.concat(
    [df.assign(DriverNumber=driver) for driver, df in session.car_data.items()],
    ignore_index=True
)
cols = ['DriverNumber'] + [col for col in telemetry.columns if col != 'DriverNumber']
telemetry = telemetry[cols]
telemetry.to_csv('telemetry.csv', index=False)

# Data Cleaning

during data tables review, it was found that we have certain columns in the 'timedelta64' data type,
however we cannot use that for analysis so we are processing it to 'time' data type.

In [12]:
from datetime import time

def timedelta_to_time(x):
    if pd.isnull(x) or not isinstance(x, pd.Timedelta) or x is pd.NaT:
        return None
    try:
        total_sec = int(x.total_seconds())
        hours = total_sec // 3600
        minutes = (total_sec % 3600) // 60
        seconds = total_sec % 60
        milliseconds = round(x.microseconds / 1000)

        if hours > 0:
            return f"{hours}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}"
        else:
            return f"{minutes}:{seconds:02d}.{milliseconds:03d}"
    except (ValueError, AttributeError):
        return None


def convert_columns_to_time(df, columns):
    df = df.copy()
    for col in columns:
        df[f'{col}_Time'] = df[col].apply(timedelta_to_time)
    return df

# manually selected timedelta columns

timedelta_columns_laps = ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'PitInTime', 'PitOutTime']
timedelta_columns_weather = ['Time']

# converting timedelta columns to time

laps = convert_columns_to_time(laps, timedelta_columns_laps)
weather = convert_columns_to_time(weather, timedelta_columns_weather)

# pulling data into csv

laps.to_csv('laps.csv', index=False)
weather.to_csv('weather.csv', index=False)